[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/experiment/finetune-check-2026-11-03/notebooks/10_finetune.ipynb)

# 10 — Fine-tune the poultry-house model, then test it on counties it never saw

**Question.** Does fine-tuning the published model (Robinson et al. 2022) on notebook 09's data find more of the houses it misses on the dark 2023 photos, without adding false detections?

**How.**
1. Start from the published weights (same architecture, same input), train on 09's **train** counties (Worcester, Somerset, Caroline, Kent, Talbot, Cecil + the western not-poultry examples from 07) with random crops, rotations, rescaling and brightness changes; 2023 photos drawn twice as often.
2. After every round (epoch) score house pixels on **Dorchester** (validation); keep the best round. Round 0 = the published model.
3. **The real test:** run the whole notebook-05 pipeline (same tiles, same 2023 images, same shape filter) on **Wicomico and Queen Anne's** with the new model, and compare with notebook 05's run of the published model and with the final v1 map (published model + outline/SAM/brightness fixes). Scored on the USGS tracing (houses found) and notebook 07's labels (detections confirmed real or rejected); new detections nobody has checked yet are counted both ways (precision range) and shown at the end.

Note: the published model was trained on all of Delmarva on older photos, so Wicomico and Queen Anne's are "unseen" only for the 2023 photos and for our fine-tuning — that's the comparison that matters here.

**Needs a GPU** (Runtime → Change runtime type → T4 GPU). Training ~30–60 min; the county test ~1–1.5 h (or ~20 min with `TEST_TILES = "traced"`). Resumable: Run all again after a crash.
**Outputs.** `Task1_results/models/finetune_v1/` (`best.pt`, `history.csv`, `finetune_meta.json`) and `Task1_results/experiments/finetune_v1/` (per-county runs, `comparison.csv`). The published model is never changed.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "experiment/finetune-check-2026-11-03"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, tiles, region, viz
from geo_anom.task1 import county as C
from geo_anom.task1.kmz import write_kmz
from geo_anom.task1.pipeline import run_pipeline
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth
from geo_anom.task1 import review as R
from geo_anom.task1 import training as T
from geo_anom.task1 import finetune as F, detect
from geo_anom.task1.refine import rectangle
from shapely.geometry import box
from geo_anom.task1.pipeline import git_commit
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
RUN = "finetune_v1"
DATA = env.results_root() / "training" / "v1"           # notebook 09's output
MODEL_DIR = env.results_root() / "models" / RUN
EXP = env.results_root() / "experiments" / RUN
EPOCHS = 15
BATCH = 16
SAMPLES_PER_EPOCH = 4000          # random crops per round
LR, ENCODER_LR = 3e-4, 1e-4       # decoder / encoder learning rates
YEAR_WEIGHT = {"2023": 2.0}       # draw 2023 windows twice as often
TEST_COUNTIES = T.SPLITS["test"]  # Wicomico, Queen Anne's
TEST_TILES = "all"                # "all" county tiles (honest precision) or "traced" (only tiles holding a traced house; faster)
MAX_WORKERS = 12
ROAD_FILTER_M = 20                # in the comparison, drop detections whose centre is this close to a public road (None = off)
import torch
DEVICE = detect.get_device()
print("device:", DEVICE, "| data:", DATA, "| model ->", MODEL_DIR)
if DEVICE.type != "cuda":
    print("No GPU: training will be very slow. In Colab: Runtime -> Change runtime type -> T4 GPU.")

## 1. Training data

In Colab the windows are copied from the shared drive to the session's own disk first (reading thousands of small files from Drive during training is slow). Only the copy is read; nothing on the drive changes.

In [ ]:
import shutil
plan = gpd.read_file(DATA / "plan.geojson"); plan["year"] = plan["year"].astype(str)
LOCAL = Path("/content/training_v1") if IN_COLAB else DATA
if IN_COLAB:
    for split in ["train", "val"]:
        src = DATA / split
        if src.exists():
            n_src = sum(1 for _ in src.rglob("*.npz")); dst = LOCAL / split
            n_dst = sum(1 for _ in dst.rglob("*.npz")) if dst.exists() else 0
            if n_dst < n_src:
                print(f"copying {split}: {n_src} files ..."); shutil.copytree(src, dst, dirs_exist_ok=True)
train_loader, val_loader = F.loaders(LOCAL, plan, batch_size=BATCH, samples_per_epoch=SAMPLES_PER_EPOCH,
                                     year_weight=YEAR_WEIGHT, workers=2)
print(f"train windows {len(train_loader.dataset)} | val windows {len(val_loader.dataset)}")
display(plan.pivot_table(index="split", columns="year", values="window_id", aggfunc="count", fill_value=0))

## 2. Train (resumable)

In [ ]:
model, _ = detect.load_model(device=DEVICE)          # the published weights, SHA-256 checked
hist = F.train(model, DEVICE, train_loader, val_loader, MODEL_DIR, epochs=EPOCHS, lr=LR, encoder_lr=ENCODER_LR,
               meta={"base": env.CHECKPOINT["source"], "base_sha256": env.CHECKPOINT["sha256"], "data": str(DATA),
                     "git_commit": git_commit()})
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
h = hist[hist.epoch > 0]
ax[0].plot(h.epoch, h.train_loss, "o-"); ax[0].set_title("training loss"); ax[0].set_xlabel("round")
for k, c in [("val_f1", "k"), ("val_precision", "#2b7a3d"), ("val_recall", "#b03a2e")]:
    ax[1].plot(hist.epoch, hist[k], "o-", color=c, label=k[4:])
ax[1].axhline(hist.val_f1.iloc[0], color="0.6", ls="--", label="published model F1")
ax[1].set_title("Dorchester (validation), house pixels"); ax[1].set_xlabel("round"); ax[1].legend()
plt.show()
display(hist.round(3))

## 3. The test: whole held-out counties, whole pipeline

Same tiles and the same 2023 images as notebook 05 (its recorded NAIP item IDs), same shape filter; only the model changes.

In [ ]:
BEST = MODEL_DIR / "best.pt"
assert BEST.exists(), "no fine-tuned model beat the published one on validation -- nothing to test"
print("testing", BEST, "| sha256", env.sha256(BEST)[:16])
from tqdm.auto import tqdm
gt = load_ground_truth().to_crs(region.UTM)

def tile_box(s):
    return gpd.GeoSeries([box(*s["bbox"])], crs=4326).to_crs(region.UTM).iloc[0]

selected = {}
for n in TEST_COUNTIES:
    sites = json.loads((C.county_out(n) / "sites.json").read_text())
    if TEST_TILES == "traced":
        cu = region.county_boundary(n).to_crs(region.UTM).unary_union
        g = gt[gt.centroid.within(cu)].centroid
        sites = [s for s in sites if s.get("bbox") and g.within(tile_box(s)).any()]
    selected[n] = sites
    print(n, len(sites), "tiles")
for n, sites in selected.items():
    out = EXP / C.county_out(n).name
    print(f"\n=== {n} ===")
    try:
        run_pipeline(sites, out, manifest=tiles.load_registry(), manifest_path=tiles.REGISTRY_SNAPSHOT,
                     prefer="area", stream=True, max_workers=MAX_WORKERS, progress=tqdm,
                     filters={"farm": [], "county": [n]}, checkpoint=BEST)
    except Exception as e:
        print(f"{n}: FAILED ({type(e).__name__}: {e}) -- run again to retry")
    finally:
        import gc; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

## 4. Before vs after

- **published model** = notebook 05's run; **final map v1** = published model + rectangle outlines + SAM repair + brightened-photo rerun (the 2,823-house map); **fine-tuned** = this model, plain pipeline (rectangle outlines applied for area).
- *found* = USGS-traced houses touched; *real* = detections on a traced or 07-confirmed house; *wrong* = on a 07-rejected detection; *unchecked* = neither (precision low counts them wrong, high counts them right).
- After the first run, check the *unchecked* ones in **notebook 07 section 7**, then rerun this section (not the test): those labels count here too.
- **Road rule** (`ROAD_FILTER_M`, on by default): detections whose centre is within 20 m of an OpenStreetMap public road are dropped from every model before scoring (the Tulbure road rule, applied after the run). The fine-tuned model marks stretches of road; the published model doesn't. *dropped_on_road* counts them.


In [ ]:
confirmed, rejected, _ = T.review_polygons(env.results_root() / "review", C.all_counties_in_order())
# + notebook 07 section 7: this run's own new detections, checked by eye
CHECKED = env.results_root() / "review" / RUN
for name in ["confirmed", "rejected"]:
    f = CHECKED / f"{name}.geojson"
    if f.exists():
        extra = gpd.read_file(f).to_crs(region.UTM)[["geometry"]]
        if name == "confirmed": confirmed = gpd.GeoDataFrame(pd.concat([confirmed, extra], ignore_index=True), crs=region.UTM)
        else: rejected = gpd.GeoDataFrame(pd.concat([rejected, extra], ignore_index=True), crs=region.UTM)
        print(f"+ {len(extra)} {name} from 07 section 7")
FINAL = env.results_root() / "final_eastern_shore" / "eastern_shore_houses_final.geojson"
final = gpd.read_file(FINAL).to_crs(region.UTM) if FINAL.exists() else None
rows, unchecked = [], {}
for n, sites in selected.items():
    cty = region.county_boundary(n); cu = cty.to_crs(region.UTM).unary_union
    area = cu if TEST_TILES == "all" else gpd.GeoSeries([tile_box(s) for s in sites], crs=region.UTM).unary_union.intersection(cu)
    out = EXP / C.county_out(n).name
    if not (out / "buildings.geojson").exists():
        print(n, "not run yet"); continue
    runs = {"published model": C.county_buildings(C.load_county_run(C.county_out(n))[2], cty).to_crs(region.UTM),
            "fine-tuned": C.county_buildings(C.load_county_run(out)[2], cty).to_crs(region.UTM)}
    if final is not None:
        runs["final map v1"] = final[final.county == n]
    for label, b in runs.items():
        b = b[["geometry"]].copy(); b["geometry"] = b.geometry.apply(rectangle)
        on_road = 0
        if ROAD_FILTER_M:
            near = detect.distance_to_roads(b, env.results_root() / "reference" / "osm_roads") < ROAD_FILTER_M
            on_road, b = int((near & b.centroid.within(area)).sum()), b[~near]
        s = F.score_area(b.reset_index(drop=True), gt, area, confirmed, rejected)
        if label == "fine-tuned":
            bb = b[b.centroid.within(area)].reset_index(drop=True)
            unchecked[n] = bb.loc[s.get("_unchecked_index", [])]
        s.pop("_unchecked_index", None)
        rows.append({"county": n, "model": label, **s, "dropped_on_road": on_road, "roof_km2": round(b[b.centroid.within(area)].area.sum() / 1e6, 3)})
comp = pd.DataFrame(rows)
tot = comp.groupby("model")[["traced", "detections", "found", "real", "wrong", "unchecked", "dropped_on_road", "roof_km2"]].sum()
tot["recall"] = tot.found / tot.traced
tot["precision_low"] = tot.real / tot.detections
tot["precision_high"] = (tot.real + tot.unchecked) / tot.detections
order = [m for m in ["published model", "final map v1", "fine-tuned"] if m in tot.index]
display(comp.round(3)); display(Markdown("**Both test counties**")); display(tot.loc[order].round(3))
EXP.mkdir(parents=True, exist_ok=True); comp.to_csv(EXP / "comparison.csv", index=False)

## 5. Detections nobody has checked yet

New houses the fine-tuned model finds that are neither traced nor in notebook 07. If they look like poultry houses (new since 2016/17), the high end of the precision range is the right one; if not, the low end.

In [ ]:
def show_unchecked(n, k=12):
    u = unchecked.get(n)
    if u is None or not len(u):
        print(n, ": none"); return
    cols = 4; rows_ = int(np.ceil(min(k, len(u)) / cols))
    fig, ax = plt.subplots(rows_, cols, figsize=(4 * cols, 4 * rows_), squeeze=False)
    for a, (_, r) in zip(ax.flat, u.head(k).iterrows()):
        c = gpd.GeoSeries([r.geometry.centroid], crs=region.UTM).to_crs(4326).iloc[0]
        got = tiles.fetch_window(c.x, c.y, 300)
        if got is None: continue
        img, t, crs = got
        a.imshow(img[..., :3], extent=(t.c, t.c + t.a * img.shape[1], t.f + t.e * img.shape[0], t.f))
        g = gpd.GeoSeries([r.geometry], crs=region.UTM).to_crs(crs).iloc[0]
        a.plot(*g.exterior.xy, color="yellow", lw=1.5)
    for a in ax.flat: a.axis("off")
    fig.suptitle(f"{n}: {len(u)} unchecked detections (first {min(k, len(u))})"); plt.tight_layout(); plt.show()

for n in selected:
    show_unchecked(n)

## 6. Save

In [ ]:
meta = {"run": RUN, "made_at": pd.Timestamp.now().isoformat(timespec="seconds"), "git_commit": git_commit(),
        "checkpoint": str(BEST), "checkpoint_sha256": env.sha256(BEST),
        "best_epoch": int(torch.load(BEST, map_location="cpu")["epoch"]),
        "settings": {"epochs": EPOCHS, "batch": BATCH, "samples_per_epoch": SAMPLES_PER_EPOCH, "lr": LR,
                     "encoder_lr": ENCODER_LR, "year_weight": YEAR_WEIGHT, "test_tiles": TEST_TILES},
        "data": str(DATA), "splits": T.SPLITS,
        "validation": hist.to_dict("records"),
        "test": tot.reset_index().to_dict("records")}
F.save_meta(MODEL_DIR / "finetune_meta.json", **meta)
print("saved", MODEL_DIR / "finetune_meta.json")

## What this shows

*(Write after the run: validation F1 by round vs the published model; on Wicomico + Queen Anne's, houses found / precision range / roof area for the published model, the final v1 map and the fine-tuned model; what the unchecked detections look like.)*

**Decision rule (set before looking):** adopt the fine-tuned model for the v2 map if, on the test counties, it finds more traced houses than the published model **and** its low-end precision is no more than 1 point below the published model's. Then rerun notebook 05 on all Eastern Shore counties with `checkpoint=` this model, re-apply the outline fixes, and record the checkpoint's SHA-256 in `env.py`.